In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

# importerer data
stations_df = pd.read_csv("../raw_data/stations.csv")
trips_df = pd.read_csv("../raw_data/trips.csv.gz")
weather_df = pd.read_csv("../raw_data/weather.csv")

In [28]:
# korrigerer til riktig tidssone før videre analyse
# PS - vil endre dtype på timestamp til Europe/Oslo
stations_df["timestamp"] = pd.to_datetime(stations_df["timestamp"], utc=True).dt.tz_convert("Europe/Oslo")
trips_df["started_at"] = pd.to_datetime(trips_df["started_at"], utc=True, format="ISO8601").dt.tz_convert("Europe/Oslo")
trips_df["ended_at"] = pd.to_datetime(trips_df["ended_at"], utc=True, format="ISO8601").dt.tz_convert("Europe/Oslo")
weather_df["timestamp"] = pd.to_datetime(weather_df["timestamp"], utc=True).dt.tz_convert("Europe/Oslo")

In [29]:
# OpenAI's ChatGPT ble brukt til å formulere og/eller komme opp med ideer for deler av kode.
# Dette gjelder target_stations_df, der det blir brukt resampling ihht til LOCF-prinsippet,
# opprettelsen av de tre datasettene fra trips, merge-funksjonene for å smelte sammen datasettene
# og interpolering og utfylling av NaN-rader til å inneholde 0 for analyse i del 2.
# All kode som er hjulpet av AI er forstått i seg selv og ihht til resten av programmet.
# OpenAI - https://chatgpt.com - henta 14.09.25 


# preprosesserer stations_df

# velger ut stasjonene vi ønsker å predikere og sorterer
target_stations = np.array(["Møllendalsplass", "Torgallmenningen", "Grieghallen",
                            "Høyteknologisenteret", "Studentboligene", "Akvariet",
                            "Damsgårdsveien 71", "Dreggsallmenningen Sør", "Florida Bybanestopp"])

filtered_stations = stations_df[stations_df["station"].isin(target_stations)]
filtered_stations = filtered_stations.set_index("timestamp").sort_index()

# runder opp til neste hele time ihht locf og sorterer
target_stations_df = filtered_stations.groupby("station")["free_bikes"].apply(lambda s: s.resample("1h").ffill()
                                        ).reset_index().pivot(index="timestamp", columns="station",
                                        values="free_bikes").sort_index()
target_stations_df.columns.name = None
target_stations_df = target_stations_df.reset_index()

# bruker melt-funksjonen for å smelte tilbake til ønsket format
target_stations_df = target_stations_df.melt(id_vars=["timestamp"], var_name="station", value_name="free_bikes")


# preprosesserer trips_df

# sorterer ut timer fra timestamp
trips_hourly = trips_df.groupby(trips_df["started_at"].dt.floor("h")).size().reset_index(name="trips_count")
trips_hourly = trips_hourly.rename(columns={"started_at": "timestamp"})

# finner mengde ankomster og avreiser per time i data
trips_arrivals = trips_df.groupby([trips_df["ended_at"].dt.floor("h"), "end_station_name"]).size().reset_index(name="arrivals") #(["end_station_name", "ended_at"]).size().reset_index(name="arrivals")
trips_arrivals = trips_arrivals.rename(columns={"end_station_name":"station", "ended_at":"timestamp"})

trips_departures = trips_df.groupby([trips_df["started_at"].dt.floor("h"), "start_station_name"]).size().reset_index(name="departures")#(["start_station_name", "started_at"]).size().reset_index(name="departures")
trips_departures = trips_departures.rename(columns={"start_station_name":"station", "started_at":"timestamp"})

# slår sammen alt til en dataframe, merged_df, med alle stasjoner, sykkelbruk og vær.
# Slår sammen på stations.csv slik at data kun går over ønsket periode
merged_df = pd.merge_asof(target_stations_df.sort_values("timestamp"), trips_arrivals.sort_values("timestamp"),
                          by="station", left_on="timestamp", right_on="timestamp", direction="backward")
merged_df = pd.merge_asof(merged_df.sort_values("timestamp"), trips_departures.sort_values("timestamp"),
                                      by="station", left_on="timestamp", right_on="timestamp", direction="backward")
merged_df = merged_df.merge(weather_df, on="timestamp", how="left")
merged_df = merged_df.merge(trips_hourly, on="timestamp", how="left")

# legger til nye kolonner for prediksjonslabel og utleding av timestamp
merged_df["free_bikes_next_hour"] = (merged_df.groupby("station")["free_bikes"].shift(-1))
merged_df["hour"] = merged_df["timestamp"].dt.hour
merged_df["day_of_week"] = merged_df["timestamp"].dt.day_name()
merged_df["month"] = merged_df["timestamp"].dt.month

# imputerer data ved å fylle NaN-verdier med 0 og interpolering der nødvendig
merged_df = merged_df.dropna(subset=["free_bikes_next_hour"])
merged_df["free_bikes"] = merged_df["free_bikes"].fillna(0).astype(int)
merged_df["arrivals"] = merged_df["arrivals"].fillna(0).astype(int)
merged_df["departures"] = merged_df["departures"].fillna(0).astype(int)
merged_df["trips_count"] = merged_df["trips_count"].fillna(0).astype(int)
merged_df["temperature"] = merged_df["temperature"].interpolate()
merged_df["precipitation"] = merged_df["precipitation"].fillna(0).astype(int)
merged_df["wind_speed"] = merged_df["wind_speed"].interpolate()

# omsorterer kolonnene i merged_df
cols = [
    "timestamp",
    "hour",
    "day_of_week",
    "month",
    "station",
    "free_bikes",
    "free_bikes_next_hour",
    "arrivals",
    "departures",
    "trips_count",
    "temperature",
    "precipitation",
    "wind_speed"
]
merged_df = merged_df[cols]

# dropper målinger fra ca midt i april 2024 til ca midt i august 2024 grunnet statiske målinger i perioden
mask = ~((merged_df["timestamp"] >= "2024-04-15") & (merged_df["timestamp"] <= "2024-08-15"))
merged_df = merged_df[mask]

# bruker indeksering for å kun bruke treningsdata (første 70%) i analyse
split_index = int(0.7*len(merged_df))
merged_df = merged_df.iloc[: split_index]

print(merged_df.info())
print(merged_df.describe())

<class 'pandas.core.frame.DataFrame'>
Index: 40029 entries, 0 to 66389
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype                      
---  ------                --------------  -----                      
 0   timestamp             40029 non-null  datetime64[ns, Europe/Oslo]
 1   hour                  40029 non-null  int32                      
 2   day_of_week           40029 non-null  object                     
 3   month                 40029 non-null  int32                      
 4   station               40029 non-null  object                     
 5   free_bikes            40029 non-null  int64                      
 6   free_bikes_next_hour  40029 non-null  float64                    
 7   arrivals              40029 non-null  int64                      
 8   departures            40029 non-null  int64                      
 9   trips_count           40029 non-null  int64                      
 10  temperature           40029 non-null  f

In [30]:
merged_df.head()

,timestamp,hour,day_of_week,month,station,free_bikes,free_bikes_next_hour,arrivals,departures,trips_count,temperature,precipitation,wind_speed
0,2024-04-10 23:00:00+02:00,23,Wednesday,4,Akvariet,0,0.0,3,3,14,5.1,0,17.4
1,2024-04-10 23:00:00+02:00,23,Wednesday,4,Dreggsallmenningen Sør,0,12.0,1,1,14,5.1,0,17.4
2,2024-04-10 23:00:00+02:00,23,Wednesday,4,Studentboligene,0,15.0,1,1,14,5.1,0,17.4
3,2024-04-10 23:00:00+02:00,23,Wednesday,4,Florida Bybanestopp,0,0.0,1,1,14,5.1,0,17.4
4,2024-04-10 23:00:00+02:00,23,Wednesday,4,Møllendalsplass,0,4.0,1,1,14,5.1,0,17.4


In [31]:
# visualiserer sammenhengen mellom temperatur og ledige sykler på Grieghallen-stasjon via lineplot
merged_df_grieghallen = merged_df[merged_df["station"]=="Grieghallen"]

fig = px.line(merged_df_grieghallen, "timestamp", ["temperature", "free_bikes"])
fig.update_layout(title="temperatur vs ledige sykler på Grieghallen-stasjon",
              yaxis_title="ledige sykler / temperatur")
fig.show()

In [32]:
# visualiserer sykkelbruk og temperatur over tid via lineplot
fig = px.line(merged_df, "timestamp", ["trips_count", "temperature"])
fig.update_layout(title="sykkelbruk og temperatur over tid", yaxis_title="antall turer / temperatur")
fig.show()

In [33]:
# visualiserer sammenheng mellom sykkelbruk og temperatur via scatter
fig = px.scatter(merged_df, "temperature", "trips_count")
fig.update_layout(title="sykkelbruk vs temperatur", xaxis_title="temperatur (C)",
                  yaxis_title="antall turer")
fig.show()

In [34]:
# visualiserer sammenheng mellom sykkelbruk og vind via scatter
fig = px.scatter(merged_df, "wind_speed", "trips_count")
fig.update_layout(title="sykkelbruk vs vind", xaxis_title="vindfart (m/s)",
                  yaxis_title="antall turer")
fig.show()

In [35]:
# visualiserer sammenheng mellom sykkelbruk og nedbør (når det er målt) via scatter
fig = px.scatter(merged_df, "precipitation", "trips_count")
fig.update_layout(title="sykkelbruk vs nedbør", xaxis_title="nedbør",
                  yaxis_title="antall turer")
fig.show()

In [36]:
# visualiserer ledige sykler per time på døgnet via boxplot
fig = px.box(merged_df, "hour", "free_bikes", color="station")
fig.update_layout(title="ledige sykler per time på døgnet", xaxis_title="time på døgnet",
                  yaxis_title="ledige sykler")
fig.show()

In [37]:
# visualiserer neste sykkel per time på døgnet (prediksjonslabel ihht LOCF)
fig = px.box(merged_df, "hour", "free_bikes_next_hour", color="station")
fig.update_layout(title="predikert ledige sykler per time på døgnet", xaxis_title="time på døgnet",
             yaxis_title="ledige sykler (predikert)")
fig.show()

In [38]:
# visualiserer ledige sykler per ukedag via boxplot
fig = px.box(merged_df, "day_of_week", "free_bikes", color="station")
fig.update_layout(title="ledige sykler per ukedag", xaxis_title="dag i uka",
                  yaxis_title="ledige sykler")
fig.show()